# Predykcja zakupu po reklamie społecznościowej — projekt z Uczenia Maszynowego

> Hubert Śliwiński gr. 2 nr 32532

## Link do projektu
- Github: https://github.com/SliskiPlumek/projekt-ml-smads
- Google Colab: https://colab.research.google.com/drive/1fD6GqeMK8xh1kBzAsX3HOB0200vUzXLM?usp=sharing

## Opis projektu
Projekt dotyczy przewidywania, **czy użytkownik kupił reklamowany produkt po obejrzeniu
reklamy w sieci społecznościowej** (`Purchased`: 1 = kupił, 0 = nie kupił). Jest to
zadanie **klasyfikacji binarnej** — marketingowy problem „reklama → konwersja".

**Dane:** publiczny zbiór z Kaggle — *Social Network Ads* (`rakeshrau/social-network-ads`),
ładowany biblioteką **`kagglehub`**.

**Przebieg analizy:** wczytanie danych → opis → czyszczenie → porównanie dwóch modeli
→ szczegółowa ocena wybranego modelu → wizualizacje → wnioski.


## 0. Import bibliotek

W pierwszym kroku ładowane są biblioteki do obliczeń, wykresów oraz uczenia maszynowego.

In [ ]:
# --- Biblioteki do obliczeń i danych ---
import numpy as np               # operacje numeryczne
import pandas as pd              # tabele danych (DataFrame)

# --- Biblioteki do wykresów ---
import matplotlib.pyplot as plt  # podstawowe wykresy
import seaborn as sns            # ładniejsze wykresy statystyczne

# --- Narzędzia uczenia maszynowego (scikit-learn) ---
from sklearn.model_selection import train_test_split   # podział na zbiór treningowy/testowy
from sklearn.tree import DecisionTreeClassifier         # pojedyncze drzewo decyzyjne
from sklearn.ensemble import RandomForestClassifier     # las losowy (zespół drzew)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Ustalone ziarno losowości -> powtarzalność wyników przy każdym uruchomieniu
np.random.seed(42)
sns.set_theme(style="whitegrid")   # styl tła wykresów
print("Biblioteki zaimportowane.")


## 1. Wczytanie danych z Kaggle

Dane pobierane są wprost z Kaggle biblioteką `kagglehub` (zbiór
`rakeshrau/social-network-ads`). To **publiczny** zbiór, więc pobranie działa **bez logowania**.

In [ ]:
import kagglehub   # biblioteka do pobierania zbiorów z Kaggle
import os            # operacje na ścieżkach i plikach

# Pobranie zbioru; funkcja zwraca ścieżkę do folderu z pobranymi plikami
sciezka = kagglehub.dataset_download("rakeshrau/social-network-ads")

# Odnalezienie pliku CSV w pobranym folderze
plik = [f for f in os.listdir(sciezka) if f.endswith(".csv")][0]

# Wczytanie pliku CSV do tabeli (DataFrame)
df = pd.read_csv(os.path.join(sciezka, plik))

print("Wczytano zbiór:", plik)
print("Wymiary zbioru:", df.shape)   # (liczba wierszy, liczba kolumn)
df.head()                            # podgląd pierwszych 5 wierszy


## 2. Opis danych

Zbiór zawiera informacje o **400 użytkownikach** sieci społecznościowej oraz to,
czy kupili reklamowany produkt.

| Kolumna | Opis |
|---------|------|
| `User ID` | identyfikator użytkownika (nieistotny dla predykcji) |
| `Gender` | płeć (Male / Female) |
| `Age` | wiek |
| `EstimatedSalary` | szacowane roczne wynagrodzenie |
| `Purchased` | **cel:** 1 = kupił po reklamie, 0 = nie kupił |

In [ ]:
df.info()   # lista kolumn, typy danych i liczba niepustych wartości

print("\nStatystyki cech liczbowych:")
print(df.describe().round(1))   # średnia, min, max, percentyle

print("\nRozkład klasy docelowej (Purchased):")
print(df["Purchased"].value_counts())                          # liczba zer i jedynek
print("Udział kupujących:", round(df["Purchased"].mean()*100, 1), "%")  # odsetek jedynek


## 3. Czyszczenie i przygotowanie danych

Na tym etapie wykonywane są cztery czynności:
- sprawdzenie **braków danych** i **duplikatów**,
- usunięcie kolumny **`User ID`** (identyfikator nie niesie informacji predykcyjnej),
- zakodowanie tekstowej kolumny **`Gender`** na wartość liczbową (one-hot),
- podział danych na **zbiór treningowy (75%)** i **testowy (25%)**.

Podział na dwa zbiory jest istotny: model uczy się na danych treningowych, a oceniany
jest na testowych, których wcześniej nie widział — to pozwala uczciwie zmierzyć skuteczność.

In [ ]:
# 1) Kontrola jakości danych
print("Braki danych:", int(df.isna().sum().sum()))   # łączna liczba pustych pól
print("Duplikaty:", int(df.duplicated().sum()))      # liczba powtórzonych wierszy

# 2) Usunięcie identyfikatora
dane = df.drop(columns=["User ID"])

# 3) Cechy wejściowe (X) i cel (y).
#    get_dummies zamienia tekst "Male/Female" na kolumnę liczbową Gender_Male (0/1)
X = pd.get_dummies(dane.drop(columns="Purchased"), drop_first=True)
y = dane["Purchased"]
print("\nCechy użyte w modelu:", list(X.columns))

# 4) Podział na zbiór treningowy i testowy; stratify -> zachowanie proporcji klas
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print("Trening:", X_train.shape[0], "| Test:", X_test.shape[0], "próbek")


## 4. Porównanie modeli

Porównane zostają **dwa modele drzewiaste**, oba uczone na tym samym zbiorze treningowym
i oceniane na tym samym zbiorze testowym.

| Model | Jak działa | Cecha charakterystyczna |
|-------|-----------|-------------------------|
| Drzewo decyzyjne | jedno drzewo zadające serię pytań **tak/nie** dzielących dane | proste i interpretowalne, ale łatwo się przeucza |
| Las losowy | **wiele** drzew głosujących, wynik jest uśredniany | stabilniejszy, ogranicza przeuczenie, podaje ważność cech |

**Kluczowa różnica:** las losowy to po prostu **wiele** drzew decyzyjnych, których głosy
są uśredniane. Pojedyncze drzewo łatwo „uczy się na pamięć" danych treningowych
(przeuczenie), a uśrednienie wielu drzew daje stabilniejsze i zwykle dokładniejsze
przewidywania. Porównanie pokazuje, co dokładnie zyskuje się dzięki zastosowaniu lasu
zamiast pojedynczego drzewa.

In [ ]:
# Dwa modele drzewiaste (żaden nie wymaga skalowania cech)
modele = {
    "Drzewo decyzyjne": DecisionTreeClassifier(random_state=42),
    "Las losowy":       RandomForestClassifier(n_estimators=200, random_state=42),
}

# Trening każdego modelu i pomiar dokładności na zbiorze testowym
wyniki = []
for nazwa, m in modele.items():
    m.fit(X_train, y_train)                            # trening na danych treningowych
    acc = accuracy_score(y_test, m.predict(X_test))    # dokładność na danych testowych
    wyniki.append({"Model": nazwa, "Dokładność (test)": round(acc, 3)})

# Tabela porównawcza, posortowana od najlepszego modelu
porownanie = pd.DataFrame(wyniki).sort_values("Dokładność (test)", ascending=False).reset_index(drop=True)
print(porownanie.to_string(index=False))


In [ ]:
# Wizualizacja porównania — wykres słupkowy dokładności obu modeli
plt.figure(figsize=(6, 3))
sns.barplot(data=porownanie, x="Dokładność (test)", y="Model",
            palette="viridis", hue="Model", legend=False)
plt.title("Porównanie modeli — dokładność na zbiorze testowym")
plt.xlim(0.7, 1.0)
plt.tight_layout(); plt.show()


**Las losowy** osiąga wyższą dokładność niż pojedyncze **drzewo decyzyjne** —
potwierdza to, że uśrednienie wielu drzew daje stabilniejszy wynik. Dlatego do dalszej,
szczegółowej analizy wybrany zostaje **las losowy**.

## 5. Szczegółowa ocena wybranego modelu (las losowy)

Wybrany model oceniany jest na zbiorze testowym: liczona jest dokładność, raport
klasyfikacji (precyzja, czułość, F1) oraz macierz pomyłek.

In [ ]:
# Do dalszej analizy wybrano LAS LOSOWY (wytrenowany już w sekcji 4)
model = modele["Las losowy"]

y_pred = model.predict(X_test)              # predykcje dla zbioru testowego
acc = accuracy_score(y_test, y_pred)        # dokładność
print(f"Dokładność (accuracy): {acc:.3f}  ({acc*100:.1f}%)\n")

print("Raport klasyfikacji:")
print(classification_report(y_test, y_pred, target_names=["nie kupił","kupił"], digits=3))

print("Macierz pomyłek:")
print(confusion_matrix(y_test, y_pred))


## 6. Wizualizacje

### 6.1. Kto kupuje? — wiek vs zarobki
Każdy punkt to użytkownik; kolor oznacza, czy dokonał zakupu.

In [ ]:
# Wykres punktowy: oś X = wiek, oś Y = zarobki, kolor = czy nastąpił zakup
plt.figure(figsize=(8, 5.5))
sns.scatterplot(data=df, x="Age", y="EstimatedSalary", hue="Purchased",
                palette={0: "#9aa6b2", 1: "#2ca02c"}, alpha=0.8)
plt.title("Zakup w zależności od wieku i zarobków")
plt.xlabel("Wiek"); plt.ylabel("Szacowane wynagrodzenie")

# Poprawne podpisy legendy (zgodne z kolorami punktów)
uchwyty, _ = plt.gca().get_legend_handles_labels()
plt.legend(uchwyty, ["nie kupił", "kupił"], title="Zakup")
plt.tight_layout(); plt.show()


### 6.2. Macierz pomyłek
Te same wartości co w sekcji 5, przedstawione graficznie — ciemna przekątna to trafienia.

In [ ]:
# Macierz pomyłek w formie mapy cieplnej
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["nie kupił","kupił"], yticklabels=["nie kupił","kupił"])
plt.xlabel("Predykcja modelu"); plt.ylabel("Prawdziwa klasa")
plt.title("Macierz pomyłek"); plt.tight_layout(); plt.show()


### 6.3. Które cechy są najważniejsze dla modelu?
Las losowy podaje, jak bardzo każda cecha wpływała na jego decyzje.

In [ ]:
# Ważność cech wg lasu losowego (wartości sumują się do 1)
imp = pd.Series(model.feature_importances_, index=X.columns).sort_values()
plt.figure(figsize=(7, 4))
imp.plot(kind="barh", color="teal")
plt.title("Ważność cech wg modelu (las losowy)")
plt.xlabel("Ważność"); plt.tight_layout(); plt.show()
print(imp.sort_values(ascending=False).round(3))   # ranking od najważniejszej cechy


## 7. Wnioski

- Porównano dwa modele drzewiaste: **las losowy** osiągnął wyższą dokładność (ok. 90%)
  niż pojedyncze **drzewo decyzyjne** (ok. 88%), ponieważ uśrednia wiele drzew i mniej
  się przeucza.
- Wybrany **las losowy** osiąga **ok. 90% dokładności** na zbiorze testowym — bardzo dobry
  wynik jak na tak proste dane.
- O zakupie decydują przede wszystkim **wiek** i **szacowane wynagrodzenie**; **płeć**
  ma znaczenie bliskie zera.
- Na wykresie wiek–zarobki wyraźnie widać, że zakupów dokonują głównie osoby **starsze**
  oraz **młodsze, ale lepiej zarabiające**.
- **Wniosek marketingowy:** reklamy warto kierować do osób starszych i lepiej
  zarabiających; targetowanie wyłącznie po płci nie ma uzasadnienia w danych.
- **Ograniczenia:** zbiór jest niewielki (400 osób) i zawiera tylko 3 cechy, dlatego
  projekt stanowi raczej demonstrację metody niż gotowe narzędzie produkcyjne.

---

### Jak uruchomić
- **Google Colab (zalecane):** `Plik → Prześlij notatnik` → `Środowisko wykonawcze → Uruchom wszystko`.
- **Lokalnie:** `pip install scikit-learn pandas matplotlib seaborn kagglehub`, następnie uruchomienie wszystkich komórek.